# Evaluación Final - Modelo con Hard Negatives (Epoch 25)

Este notebook evalúa el mejor modelo (checkpoint_epoch_25.pth) en el test set y genera métricas para el informe final.

**Objetivo:**
- Comparar métricas: modelo original vs. modelo con hard negatives
- Validar que bajaron FPs de `knife`
- Generar tabla de comparación para informe

In [1]:
# Instalar dependencias
!pip install -q torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 14.8 MB/s eta 0:00:00a 0:00:01


In [2]:
# Montar Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Ir al repositorio
%cd /content/drive/MyDrive/procesamiento-imagenes

/content/drive/MyDrive/procesamiento-imagenes


## Preparar Test Set

Copiar el test set a disco local para evaluación rápida.

In [ ]:
# Copiar test set a disco local
print("📦 Copiando test set...")
!cp -r dataset_testing.zip /content/ 2>/dev/null || echo "(dataset_testing.zip no existe, intentando otros formatos)"
!unzip -q /content/dataset_testing.zip -d /content/ 2>/dev/null || echo "No se pudo descomprimir"

# Alternativa: si está en carpeta directa
#!ls /content/dataset_testing 2>/dev/null || echo "Test set no encontrado en /content/"

# Si está en Drive, copiar directo
#!cp -r dataset_testing /content/ 2>/dev/null || echo "Copiando desde Drive..."
#!ls -lh /content/dataset_testing/ | head -5

## Evaluar Modelo Original (Baseline)

Primero evaluamos el modelo original para tener punto de comparación.

In [ ]:
# Evaluar modelo original
print("📊 Evaluando modelo original...")
!python3 test_light_model.py \
  --model results_standard/best_model.pth \
  --images-dir /content/dataset_testing/images \
  --xml-dir /content/dataset_testing/xmls \
  --output-dir test_results_baseline \
  --confidence 0.5

## Evaluar Modelo con Hard Negatives (Epoch 25)

Ahora evaluamos el modelo mejorado.

In [ ]:
# Evaluar modelo con hard negatives (epoch 25)
print("📊 Evaluando modelo con hard negatives (epoch 25)...")
!python3 test_light_model.py \
  --model results_finetuning_negatives/checkpoint_epoch_25.pth \
  --images-dir /content/dataset_testing/images \
  --xml-dir /content/dataset_testing/xmls \
  --output-dir test_results_epoch25 \
  --confidence 0.5

## Comparar Métricas

Generar tabla comparativa con antes/después.

In [ ]:
import json
import pandas as pd

# Cargar métricas
with open('test_results_baseline/metrics.json', 'r') as f:
    baseline = json.load(f)

with open('test_results_epoch25/metrics.json', 'r') as f:
    epoch25 = json.load(f)

# Crear tabla comparativa
comparison = {
    'Métrica': ['mAP', 'mAP@50', 'mAP@75', 'Precision (knife)', 'Recall (knife)', 'F1-Score (knife)'],
    'Modelo Baseline': [
        baseline.get('mAP', 'N/A'),
        baseline.get('mAP_50', 'N/A'),
        baseline.get('mAP_75', 'N/A'),
        baseline.get('precision', {}).get('knife', 'N/A'),
        baseline.get('recall', {}).get('knife', 'N/A'),
        baseline.get('f1_score', {}).get('knife', 'N/A')
    ],
    'Con Hard Negatives (E25)': [
        epoch25.get('mAP', 'N/A'),
        epoch25.get('mAP_50', 'N/A'),
        epoch25.get('mAP_75', 'N/A'),
        epoch25.get('precision', {}).get('knife', 'N/A'),
        epoch25.get('recall', {}).get('knife', 'N/A'),
        epoch25.get('f1_score', {}).get('knife', 'N/A')
    ]
}

df = pd.DataFrame(comparison)
print("\n" + "="*80)
print("COMPARACIÓN: MODELO ORIGINAL vs. CON HARD NEGATIVES")
print("="*80)
print(df.to_string(index=False))
print("="*80)

In [ ]:
# Calcular mejoras
def calc_improvement(baseline_val, new_val):
    if isinstance(baseline_val, str) or isinstance(new_val, str):
        return "N/A"
    try:
        pct = ((new_val - baseline_val) / baseline_val) * 100
        return f"{pct:+.2f}%"
    except:
        return "N/A"

improvements = {
    'Métrica': ['mAP', 'mAP@50', 'mAP@75', 'Precision (knife)', 'Recall (knife)', 'F1-Score (knife)'],
    'Mejora': [
        calc_improvement(baseline.get('mAP'), epoch25.get('mAP')),
        calc_improvement(baseline.get('mAP_50'), epoch25.get('mAP_50')),
        calc_improvement(baseline.get('mAP_75'), epoch25.get('mAP_75')),
        calc_improvement(baseline.get('precision', {}).get('knife'), epoch25.get('precision', {}).get('knife')),
        calc_improvement(baseline.get('recall', {}).get('knife'), epoch25.get('recall', {}).get('knife')),
        calc_improvement(baseline.get('f1_score', {}).get('knife'), epoch25.get('f1_score', {}).get('knife'))
    ]
}

df_improvements = pd.DataFrame(improvements)
print("\n" + "="*80)
print("MEJORAS (% de cambio)")
print("="*80)
print(df_improvements.to_string(index=False))
print("="*80)

## Análisis de Falsos Positivos

Revisar matriz de confusión para verificar que bajaron FPs de `knife`.

In [ ]:
# Ver matriz de confusión del modelo baseline
from PIL import Image
import matplotlib.pyplot as plt
import os

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Baseline
if os.path.exists('test_results_baseline/confusion_matrix.png'):
    img_baseline = Image.open('test_results_baseline/confusion_matrix.png')
    axes[0].imshow(img_baseline)
    axes[0].set_title('Modelo Original', fontsize=14, fontweight='bold')
    axes[0].axis('off')
else:
    axes[0].text(0.5, 0.5, 'Matriz no disponible', ha='center', va='center')
    axes[0].axis('off')

# Epoch 25
if os.path.exists('test_results_epoch25/confusion_matrix.png'):
    img_epoch25 = Image.open('test_results_epoch25/confusion_matrix.png')
    axes[1].imshow(img_epoch25)
    axes[1].set_title('Con Hard Negatives (Epoch 25)', fontsize=14, fontweight='bold')
    axes[1].axis('off')
else:
    axes[1].text(0.5, 0.5, 'Matriz no disponible', ha='center', va='center')
    axes[1].axis('off')

plt.tight_layout()
plt.savefig('comparacion_matrices_confusion.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Imagen guardada: comparacion_matrices_confusion.png")

## Guardar Resultados a Drive

Copiar todos los resultados de evaluación a Drive para referencia futura.

In [ ]:
# Crear carpeta de evaluación final
!mkdir -p evaluacion_final

# Copiar resultados
!cp -r test_results_baseline evaluacion_final/
!cp -r test_results_epoch25 evaluacion_final/
!cp comparacion_matrices_confusion.png evaluacion_final/

# Guardar tabla de comparación como CSV
df.to_csv('evaluacion_final/comparacion_metricas.csv', index=False)
df_improvements.to_csv('evaluacion_final/mejoras_porcentuales.csv', index=False)

print("✅ Resultados guardados en carpeta: evaluacion_final/")

# Copiar a Drive
!cp -r evaluacion_final /content/drive/MyDrive/procesamiento-imagenes/
print("✅ Resultados copiados a Drive")

## Resumen para el Informe

Datos listos para incluir en INFORME_FINAL.md

In [ ]:
print("\n" + "="*80)
print("RESUMEN EJECUTIVO PARA INFORME FINAL")
print("="*80)
print(f"""
### Fine-tuning con Hard Negatives

**Modelo Original (Baseline):**
- mAP: {baseline.get('mAP', 'N/A')}
- mAP@50: {baseline.get('mAP_50', 'N/A')}
- mAP@75: {baseline.get('mAP_75', 'N/A')}
- Precision (knife): {baseline.get('precision', {}).get('knife', 'N/A')}
- Recall (knife): {baseline.get('recall', {}).get('knife', 'N/A')}

**Modelo con Hard Negatives (Epoch 25):**
- mAP: {epoch25.get('mAP', 'N/A')}
- mAP@50: {epoch25.get('mAP_50', 'N/A')}
- mAP@75: {epoch25.get('mAP_75', 'N/A')}
- Precision (knife): {epoch25.get('precision', {}).get('knife', 'N/A')}
- Recall (knife): {epoch25.get('recall', {}).get('knife', 'N/A')}

**Mejoras:**
- mAP: {calc_improvement(baseline.get('mAP'), epoch25.get('mAP'))}
- mAP@75: {calc_improvement(baseline.get('mAP_75'), epoch25.get('mAP_75'))}
- Precision (knife): {calc_improvement(baseline.get('precision', {}).get('knife'), epoch25.get('precision', {}).get('knife'))}
""")
print("="*80)

## Conclusión

✅ **Evaluación completa.** Los resultados están listos para el informe final.

**Carpeta con todos los resultados:**
- Drive: `/MyDrive/procesamiento-imagenes/evaluacion_final/`